# Logistat : préparation des données

**Objectif du projet :** prédire si une commande sera livrée en retard (`Late_delivery_risk` : 1 = retard, 0 = à temps).

**Ce notebook couvre uniquement la préparation :**
1. Charger le jeu d'entraînement
2. Traiter les lignes sans coordonnées
3. Garder les 15 variables retenues
4. Encoder les variables catégorielles
5. Appliquer les mêmes transformations au jeu de test, sans rien y apprendre
6. Enregistrer les jeux préparés

**Règle : on ne touche pas aux données de test.** Toute l'exploration et tous les ajustements (choix, statistiques, encodeur) se font sur l'entraînement seulement. Le test n'est chargé qu'à l'étape 5, pour y appliquer les transformations déjà décidées. `test.csv` n'est jamais modifié.

Les fichiers `train.csv` (85 %) et `test.csv` (15 %) viennent d'une séparation aléatoire stratifiée sur la cible (graine 42). Le choix des 15 variables est expliqué dans `selection_variables.docx`.

## 0. Bibliothèques

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

## 1. Chargement du jeu d'entraînement

Le fichier utilise le point-virgule `;` comme séparateur. Le jeu de test n'est pas chargé ici.

In [ ]:
train = pd.read_csv('train.csv', sep=';')

print('Entraînement :', train.shape)
print('Part de retards :', round(train['Late_delivery_risk'].mean(), 3))

## 2. Traitement des lignes sans coordonnées

Dans le fichier complet, `Latitude` et `Longitude` sont manquantes sur 220 lignes. On les étudie sur l'entraînement seulement.

In [ ]:
na = train.isna().sum()
na[na > 0]

Trois options existaient :
- **Supprimer** les lignes : on perd 220 commandes, peu au regard de 172 765.
- **Imputer** (remplacer par la médiane ou par les coordonnées moyennes de la ville du client).
- **Retirer les colonnes** `Latitude` / `Longitude`.

**Choix retenu : retirer les colonnes.** La position du client est déjà représentée par `Market` et `Order Region`, et `Latitude` / `Longitude` ne font pas partie des 15 variables. On garde ainsi les 220 commandes sans inventer de valeurs. La décision est une règle (retirer deux colonnes), elle s'appliquera telle quelle au test.

On regarde quand même si ces lignes ont un taux de retard différent, pour s'assurer qu'elles ne cachent pas un biais :

In [ ]:
sans_coord = train['Latitude'].isna()
print('Lignes sans coordonnées (train) :', sans_coord.sum())
print('Taux de retard, lignes sans coordonnées :', round(train.loc[sans_coord, 'Late_delivery_risk'].mean(), 3))
print('Taux de retard, autres lignes           :', round(train.loc[~sans_coord, 'Late_delivery_risk'].mean(), 3))

## 3. Sélection des 15 variables

On sépare les variables en deux groupes, car elles ne se traitent pas de la même façon :
- **7 catégorielles** (texte) : à encoder en nombres.
- **8 numériques** : utilisables telles quelles.

Sont exclues : la cible, les colonnes qui révèlent la réponse après coup (`delay_days`, `Delivery Status`, `Days for shipping (real)`, `shipping date`), les identifiants et les colonnes redondantes.

In [ ]:
cible = 'Late_delivery_risk'

variables_cat = ['Type', 'Shipping Mode', 'Customer Segment', 'Category Name',
                 'Department Name', 'Market', 'Order Region']
variables_num = ['Days for shipment (scheduled)', 'Order Item Product Price',
                 'Order Item Quantity', 'Order Item Discount Rate',
                 'Order Item Profit Ratio', 'order_month', 'order_weekday', 'order_hour']
variables = variables_cat + variables_num
assert len(variables) == 15

X_train, y_train = train[variables].copy(), train[cible].copy()

print('X_train :', X_train.shape)
print('Valeurs manquantes restantes :', X_train.isna().sum().sum())

Il ne reste **aucune valeur manquante** : les lignes sans coordonnées sont conservées et complètes sur les 15 variables.

Aperçu des catégories (nombre de valeurs différentes par variable) :

In [ ]:
X_train[variables_cat].nunique().sort_values()

## 4. Encodage des variables catégorielles

**One-hot ou ordinal ?**
- L'encodage **ordinal** (0, 1, 2, …) impose un ordre entre les catégories. Il n'a de sens que si un ordre naturel existe.
- L'encodage **one-hot** crée une colonne 0/1 par catégorie, sans imposer d'ordre.

Nos 7 variables catégorielles sont **nominales** (le paiement, le marché, la région ou la catégorie de produit n'ont pas d'ordre). On utilise donc le **one-hot**. Le seul candidat ordinal serait `Shipping Mode` (Same Day < First < Second < Standard), mais cet ordre est déjà porté par `Days for shipment (scheduled)` (0, 1, 2 et 4 jours), qui reste dans les variables numériques.

**Règles importantes :**
- L'encodeur est **ajusté uniquement sur le jeu d'entraînement** : la liste des catégories vient de train. Il sera ensuite appliqué au test sans être réajusté.
- `handle_unknown='ignore'` : une catégorie absente de l'entraînement mais présente dans le test reçoit des 0 partout, sans erreur.

In [ ]:
encodeur = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encodeur.fit(X_train[variables_cat])

def encoder(X):
    cat = pd.DataFrame(encodeur.transform(X[variables_cat]),
                       columns=encodeur.get_feature_names_out(variables_cat),
                       index=X.index).astype(int)
    return pd.concat([X[variables_num], cat], axis=1)

X_train_enc = encoder(X_train)

print('Avant encodage :', X_train.shape[1], 'variables')
print('Après encodage :', X_train_enc.shape[1], 'colonnes')

Les 15 variables deviennent plusieurs dizaines de colonnes, car chaque catégorie donne une colonne. Aperçu :

In [ ]:
X_train_enc.head()

**Remarque sur la mise à l'échelle :** les variables numériques ont des échelles très différentes (prix jusqu'à 2 000, quantité de 1 à 5). Une standardisation sera utile pour certains modèles (régression logistique, k plus proches voisins), mais pas pour les arbres. On la fera à l'étape de modélisation, selon le modèle choisi.

## 5. Application au jeu de test

Le test est chargé seulement maintenant. On lui applique **exactement** les transformations décidées sur l'entraînement : mêmes 15 variables, même encodeur (`transform` seulement, jamais `fit`). On n'y calcule aucune statistique et on ne modifie pas `test.csv` : le résultat va dans un nouveau fichier.

In [ ]:
test = pd.read_csv('test.csv', sep=';')
X_test, y_test = test[variables].copy(), test[cible].copy()
X_test_enc = encoder(X_test)   # encodeur ajusté sur train uniquement

assert list(X_test_enc.columns) == list(X_train_enc.columns)
print('X_test_enc :', X_test_enc.shape)

## 6. Enregistrement des jeux préparés

On enregistre les variables encodées avec la cible, pour pouvoir passer directement à la modélisation dans un prochain notebook.

In [ ]:
train_prep = X_train_enc.assign(**{cible: y_train})
test_prep = X_test_enc.assign(**{cible: y_test})

train_prep.to_csv('train_prepare.csv', sep=';', index=False)
test_prep.to_csv('test_prepare.csv', sep=';', index=False)
print('train_prepare.csv :', train_prep.shape)
print('test_prepare.csv  :', test_prep.shape)

## Résumé

| Étape | Résultat |
|---|---|
| Chargement | 146 850 lignes en entraînement, 57,3 % de retards |
| Coordonnées manquantes | Colonnes `Latitude` / `Longitude` retirées : aucune ligne perdue, aucune valeur inventée |
| Sélection | 15 variables : 7 catégorielles, 8 numériques |
| Encodage | One-hot sur les 7 catégorielles, ajusté sur l'entraînement seulement |
| Test | Chargé à la fin, transformé avec les règles de l'entraînement, `test.csv` intact |
| Sortie | `train_prepare.csv` et `test_prepare.csv` |

**Étape suivante :** modèle de référence (ci-dessous).

---
# Étape 2 : modèle de base, régression logistique

**But :** obtenir un **point de référence**. Tout modèle plus complexe (forêt aléatoire, boosting…) devra faire mieux que ce score pour valoir la peine.

**Pourquoi la régression logistique ?** Elle est simple, rapide et interprétable : chaque coefficient indique si une variable augmente ou diminue le risque de retard.

**Le test reste intact.** On évalue le modèle par **validation croisée sur le jeu d'entraînement seulement**. Le test servira une seule fois, à la toute fin, pour l'évaluation finale du modèle choisi.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, cross_val_predict
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.dummy import DummyClassifier

## 2.1 Pipeline : mise à l'échelle + modèle

La régression logistique est sensible à l'échelle des variables (prix jusqu'à 2 000, quantité de 1 à 5). On **standardise les 8 variables numériques** (moyenne 0, écart-type 1). Les colonnes one-hot (0/1) restent telles quelles.

La standardisation est placée **dans un pipeline**. Ainsi, à chaque pli de la validation croisée, la moyenne et l'écart-type sont calculés sur la partie d'apprentissage seulement, jamais sur la partie de validation.

In [ ]:
colonnes_onehot = [c for c in X_train_enc.columns if c not in variables_num]

mise_a_echelle = ColumnTransformer([
    ('num', StandardScaler(), variables_num),
    ('cat', 'passthrough', colonnes_onehot),
])

modele_logistique = Pipeline([
    ('echelle', mise_a_echelle),
    ('logit', LogisticRegression(max_iter=2000)),
])

## 2.2 Validation croisée regroupée par commande

On découpe le jeu d'entraînement en **5 plis** : le modèle apprend sur 4 et est évalué sur le 5e, cinq fois de suite.

**Pourquoi regrouper par commande ?** Une commande contient souvent plusieurs articles, qui partagent la même date, le même mode d'expédition et donc le même retard. Si des articles d'une même commande se retrouvaient à la fois en apprentissage et en validation, le score serait trop optimiste. `StratifiedGroupKFold` garde chaque commande (`Order Id`) entière dans un seul pli, tout en conservant environ 57 % de retards dans chaque pli.

In [ ]:
groupes = train['Order Id']
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

## 2.3 Métriques

- **F1** : équilibre entre précision et rappel pour la classe « retard ».
- **Rappel des retards** : part des vrais retards que le modèle détecte. C'est souvent la métrique clé, car un retard non anticipé coûte cher.
- **Précision des retards** : parmi les commandes annoncées en retard, la part qui l'est vraiment.
- **ROC-AUC** : capacité à classer une commande en retard au-dessus d'une commande à temps, indépendamment du seuil de 0,5.
- **Exactitude (accuracy)** : part de bonnes réponses, à comparer au modèle naïf ci-dessous.

In [ ]:
metriques = {'f1': 'f1', 'rappel_retard': 'recall', 'precision_retard': 'precision',
             'roc_auc': 'roc_auc', 'exactitude': 'accuracy'}

def evaluer(modele, nom):
    res = cross_validate(modele, X_train_enc, y_train, groups=groupes, cv=cv, scoring=metriques, n_jobs=-1)
    return pd.Series({m: res[f'test_{m}'].mean() for m in metriques}, name=nom)

## 2.4 Modèle naïf : le plancher

Avant tout, un modèle « naïf » qui prédit toujours la classe majoritaire (retard). Il obtient environ 57 % d'exactitude sans rien apprendre : la régression logistique doit faire nettement mieux.

In [ ]:
naif = evaluer(DummyClassifier(strategy='most_frequent'), 'Naïf (toujours retard)')
logit = evaluer(modele_logistique, 'Régression logistique')

resultats = pd.concat([naif, logit], axis=1).T.round(3)
resultats

## 2.5 Détail des prédictions (validation croisée)

Chaque commande reçoit ici une prédiction faite par un modèle qui ne l'a pas vue pendant l'apprentissage. On peut ainsi construire un rapport complet et une matrice de confusion, toujours sans toucher au test.

In [ ]:
pred_cv = cross_val_predict(modele_logistique, X_train_enc, y_train, groups=groupes, cv=cv, n_jobs=-1)

print(classification_report(y_train, pred_cv, target_names=['À temps', 'En retard'], digits=3))
ConfusionMatrixDisplay.from_predictions(y_train, pred_cv, display_labels=['À temps', 'En retard']);

## 2.6 Coefficients : quelles variables poussent vers le retard ?

On réentraîne le modèle sur tout le jeu d'entraînement pour lire ses coefficients.
- Un coefficient **positif** augmente le risque de retard.
- Un coefficient **négatif** le diminue.

Les variables numériques étant standardisées, leurs coefficients sont comparables entre eux.

In [ ]:
modele_logistique.fit(X_train_enc, y_train)

noms = modele_logistique.named_steps['echelle'].get_feature_names_out()
noms = [n.split('__', 1)[1] for n in noms]
coefs = pd.Series(modele_logistique.named_steps['logit'].coef_[0], index=noms)

print('Variables qui augmentent le plus le risque de retard :')
print(coefs.sort_values(ascending=False).head(10).round(3))
print('\nVariables qui le diminuent le plus :')
print(coefs.sort_values().head(10).round(3))

## Conclusion de l'étape 2

Le tableau de la section 2.4 donne **le score de référence**. À retenir pour le lire :
- Le modèle naïf a un **F1 élevé** uniquement parce qu'il annonce un retard partout (rappel de 100 %). Il est pourtant inutile : son **ROC-AUC vaut 0,5**, comme le hasard. Le ROC-AUC et l'exactitude sont donc les bonnes mesures pour comparer avec lui.
- La régression logistique est **très précise** quand elle annonce un retard (environ 89 %), mais elle **en rate beaucoup** (rappel d'environ 57 %). Baisser le seuil de décision sous 0,5, ou utiliser un modèle non linéaire, devrait améliorer le rappel.
- Le **mode d'expédition** domine : *First Class* augmente fortement le risque, *Same Day* et *Standard Class* le diminuent. Les prochains modèles seront évalués avec **la même validation croisée** (mêmes plis, mêmes métriques), pour une comparaison équitable.

Le jeu de test n'a pas été utilisé : il est réservé à l'évaluation finale du modèle retenu.

---
# Étape 3 : modèles plus forts

La régression logistique ne capte que des effets **linéaires** et additifs. Les modèles à base d'**arbres de décision** captent les interactions (par exemple : *First Class* **et** une certaine région) et les effets non linéaires (par exemple : l'heure de commande).

On teste deux familles :
1. **Forêt aléatoire** : beaucoup d'arbres indépendants, chacun appris sur un échantillon différent, dont on fait la moyenne des votes.
2. **Gradient boosting (LightGBM)** : des arbres construits l'un après l'autre, chacun corrigeant les erreurs des précédents. C'est souvent le plus performant sur des données tabulaires.

**Même protocole qu'à l'étape 2 :** mêmes 5 plis regroupés par commande, mêmes métriques, train seulement. Le test reste intact.

Les arbres ne sont pas sensibles à l'échelle des variables : **pas de standardisation** nécessaire, on utilise directement `X_train_enc`.

## 3.1 Forêt aléatoire

Paramètres choisis (raisonnables, pas encore optimisés) :
- `n_estimators=300` : nombre d'arbres
- `min_samples_leaf=5` : au moins 5 commandes par feuille, pour limiter le sur-apprentissage
- `n_jobs=-1` : utilise tous les cœurs du processeur
- `random_state=42` : résultats reproductibles

In [ ]:
from sklearn.ensemble import RandomForestClassifier

foret = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=42)
res_foret = evaluer(foret, 'Forêt aléatoire')
res_foret.round(3)

## 3.2 Gradient boosting : LightGBM

LightGBM est rapide et performant sur de grands jeux de données tabulaires. Installation si besoin : `pip install lightgbm`.

Paramètres choisis (raisonnables, pas encore optimisés) :
- `n_estimators=500` arbres avec un `learning_rate=0.05` : apprentissage progressif
- `num_leaves=63` : complexité de chaque arbre
- `subsample` / `colsample_bytree=0.8` : chaque arbre voit 80 % des lignes et des colonnes, ce qui réduit le sur-apprentissage

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                      subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                      random_state=42, verbose=-1)

# LightGBM n'accepte pas certains caractères dans les noms de colonnes (espaces, &, etc.)
import re
noms_propres = {c: re.sub(r'[^0-9a-zA-Z_]', '_', c) for c in X_train_enc.columns}
X_train_lgbm = X_train_enc.rename(columns=noms_propres)

res_lgbm = cross_validate(lgbm, X_train_lgbm, y_train, groups=groupes, cv=cv, scoring=metriques, n_jobs=1)
res_lgbm = pd.Series({m: res_lgbm[f'test_{m}'].mean() for m in metriques}, name='LightGBM')
res_lgbm.round(3)

## 3.3 Comparaison des modèles

Tous les scores viennent de la **même validation croisée sur l'entraînement**.

In [ ]:
comparaison = pd.concat([naif, logit, res_foret, res_lgbm], axis=1).T.round(3)
comparaison.sort_values('roc_auc', ascending=False)

In [ ]:
import matplotlib.pyplot as plt

comparaison[['roc_auc', 'exactitude', 'f1', 'rappel_retard']].plot.barh(figsize=(8, 4))
plt.xlim(0, 1)
plt.title('Comparaison des modèles (validation croisée sur train)')
plt.legend(loc='lower right')
plt.tight_layout()

## 3.4 Variables les plus importantes (LightGBM)

On réentraîne LightGBM sur tout le jeu d'entraînement pour lire l'importance des variables. On utilise le **gain** : l'amélioration totale apportée par chaque variable dans les arbres.

In [ ]:
lgbm.fit(X_train_lgbm, y_train)
importance = pd.Series(lgbm.booster_.feature_importance(importance_type='gain'),
                       index=X_train_enc.columns)
(importance / importance.sum()).sort_values(ascending=False).head(15).round(3)

## Conclusion de l'étape 3

| Modèle | ROC-AUC | Exactitude | F1 | Rappel retards | Précision retards |
|---|---|---|---|---|---|
| Forêt aléatoire | **0,771** | **0,721** | 0,708 | 0,591 | 0,884 |
| LightGBM | 0,770 | 0,714 | **0,709** | **0,608** | 0,849 |
| Régression logistique (référence) | 0,744 | 0,713 | 0,694 | 0,569 | 0,889 |
| Naïf | 0,500 | 0,573 | 0,728 | 1,000 | 0,573 |

- Les deux modèles à arbres **battent la référence** (ROC-AUC +0,027). La forêt aléatoire et LightGBM sont à égalité.
- Le gain reste **modeste** : avec ces 15 variables, une bonne part du retard semble difficile à prévoir au moment de la commande.
- LightGBM détecte un peu plus de retards (rappel 0,61) au prix d'une précision un peu plus basse.
- Les variables clés sont le **délai promis** (`Days for shipment (scheduled)`), l'**heure de commande** et le **mode d'expédition**.
- Le **rappel reste faible** (environ 60 % des retards détectés) : ajuster le seuil de décision est la prochaine piste.

Le jeu de test n'a toujours **pas** été utilisé.

**Suite :** ajuster le seuil de décision et les hyperparamètres du meilleur modèle (toujours en validation croisée sur train), puis faire l'évaluation finale, une seule fois, sur le test.

---
# Étape 4 : valider sans tricher

## Pourquoi une deuxième validation ?

La validation croisée regroupée par commande (étapes 2 et 3) empêche déjà une fuite : les articles d'une même commande ne sont jamais à la fois en apprentissage et en validation.

Mais elle **mélange les périodes** : le modèle apprend sur des commandes de 2017 pour prédire des commandes de 2015. En vrai, le modèle sera utilisé sur des **commandes futures**. Une **validation chronologique** reproduit cette situation : on apprend sur le passé, on valide sur les mois les plus récents.

**Toujours sur train seulement.** Le test n'est pas utilisé.

## 4.1 Évolution des données dans le temps (train)

Avant de couper, on regarde comment les données évoluent mois par mois : volume de commandes, taux de retard, nombre de produits et de marchés.

In [ ]:
dates = pd.to_datetime(train['order date (DateOrders)'])
mois = dates.dt.to_period('M')

evolution = train.assign(mois=mois).groupby('mois').agg(
    commandes=(cible, 'size'),
    taux_retard=(cible, 'mean'),
    produits=('Product Name', 'nunique'),
    marches=('Market', 'nunique'),
).round(3)
evolution.tail(16)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
evolution['commandes'].plot(ax=ax[0], title='Lignes de commande par mois (train)')
evolution['taux_retard'].plot(ax=ax[1], title='Taux de retard par mois (train)', ylim=(0, 1))
plt.tight_layout()

**Ce qu'on observe :**
- Le **taux de retard reste stable** (environ 57 %) sur toute la période.
- Le **volume chute** d'environ 4 300 à 1 800 lignes par mois à partir d'**octobre 2017**.
- Le **catalogue change** : de 54 produits au début à seulement 8 à 20 produits par mois à partir d'octobre 2017.
- Les **marchés** se réduisent : à partir de 2017, presque toutes les commandes viennent d'un seul marché par mois.

Les mois récents ne ressemblent donc pas tout à fait au passé. C'est exactement ce qu'une validation chronologique doit révéler.

## 4.2 Découpage chronologique

- **Apprentissage :** commandes jusqu'au **30 juin 2017**, environ 86 % de train.
- **Validation :** commandes de **juillet 2017 à janvier 2018**, environ 14 % de train, une proportion proche de celle du test.

La période de validation est elle-même découpée en deux, pour voir l'effet de la chute de volume :
- **Juillet à septembre 2017** : volume normal.
- **Octobre 2017 à janvier 2018** : volume réduit et catalogue différent.

Les commandes ne sont jamais coupées : toutes les lignes d'une commande ont la même date, donc elles tombent du même côté.

In [ ]:
coupure = pd.Timestamp('2017-07-01')
debut_reduit = pd.Timestamp('2017-10-01')

apprend = (dates < coupure).values
valid = ~apprend
valid_normal = valid & (dates < debut_reduit).values
valid_reduit = valid & (dates >= debut_reduit).values

print('Apprentissage            :', apprend.sum(), 'lignes')
print('Validation (total)       :', valid.sum(), f'lignes ({valid.mean():.1%} de train)')
print('  juil.-sept. 2017       :', valid_normal.sum())
print('  oct. 2017 - janv. 2018 :', valid_reduit.sum())

## 4.3 Entraînement sur le passé, évaluation sur les mois récents

On entraîne les trois modèles **une fois** sur la période d'apprentissage, puis on les évalue sur chaque période de validation, avec les mêmes métriques qu'avant.

In [ ]:
from sklearn.base import clone
from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score, accuracy_score

def scores(y, proba):
    pred = (proba >= 0.5).astype(int)
    return {'f1': f1_score(y, pred), 'rappel_retard': recall_score(y, pred),
            'precision_retard': precision_score(y, pred), 'roc_auc': roc_auc_score(y, proba),
            'exactitude': accuracy_score(y, pred)}

modeles_chrono = {
    'Régression logistique': (clone(modele_logistique), X_train_enc),
    'Forêt aléatoire': (clone(foret), X_train_enc),
    'LightGBM': (clone(lgbm), X_train_lgbm),
}
periodes = {'Juil. 2017 - janv. 2018': valid,
            'Juil. - sept. 2017': valid_normal,
            'Oct. 2017 - janv. 2018': valid_reduit}

lignes = []
for nom, (modele, X) in modeles_chrono.items():
    modele.fit(X[apprend], y_train[apprend])
    for periode, masque in periodes.items():
        proba = modele.predict_proba(X[masque])[:, 1]
        lignes.append({'modele': nom, 'periode': periode, **scores(y_train[masque], proba)})

chrono = pd.DataFrame(lignes).set_index(['modele', 'periode']).round(3)
chrono

## 4.4 Comparaison : validation regroupée contre validation chronologique

On met côte à côte le ROC-AUC de la validation croisée regroupée (étapes 2 et 3) et celui de la validation chronologique.

In [ ]:
roc_chrono = chrono['roc_auc'].unstack('periode')
roc_chrono.insert(0, 'CV regroupée par commande', comparaison.loc[roc_chrono.index, 'roc_auc'])
roc_chrono.round(3)

In [ ]:
roc_chrono.plot.bar(figsize=(9, 4), rot=0)
plt.ylim(0.5, 1)
plt.ylabel('ROC-AUC')
plt.title('ROC-AUC selon le type de validation')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()

## Conclusion de l'étape 4

| Modèle | CV regroupée | Juil. 2017 - janv. 2018 | Juil. - sept. 2017 | Oct. 2017 - janv. 2018 |
|---|---|---|---|---|
| Forêt aléatoire | 0,771 | **0,782** | 0,780 | 0,786 |
| LightGBM | 0,770 | **0,782** | 0,781 | 0,784 |
| Régression logistique | 0,744 | 0,755 | 0,757 | 0,755 |

- **Pas de triche détectée :** en validation chronologique, les scores ne baissent pas. Ils montent même légèrement (environ +0,01). Les scores des étapes 2 et 3 étaient donc honnêtes.
- **Les modèles tiennent dans le temps**, y compris sur la période d'octobre 2017 à janvier 2018, malgré la baisse de volume et le catalogue réduit.
- **Le classement reste le même :** forêt aléatoire et LightGBM à égalité, devant la régression logistique.

Lire le tableau 4.4 ainsi :
- Si le ROC-AUC chronologique est **proche** de celui de la validation regroupée, le modèle **se généralise dans le temps** et les scores des étapes 2 et 3 sont fiables.
- S'il **baisse nettement**, surtout sur octobre 2017 à janvier 2018, le modèle dépend de la période (produits, marchés). Il faudra alors le réentraîner régulièrement et se méfier du score obtenu sur un test aléatoire.

Le jeu de test n'a toujours **pas** été utilisé.

---
# Étape 5 : évaluer avec les bonnes métriques

## Pourquoi l'exactitude seule ne suffit pas

**57 % des commandes sont en retard.** Un modèle « paresseux » qui répond toujours « retard » obtient donc **57 % d'exactitude** sans rien apprendre. Une exactitude de 71 % paraît bonne, mais elle n'est qu'à 14 points de ce modèle inutile.

> Attention au vocabulaire : en anglais, *accuracy* (part de bonnes réponses) se traduit par **exactitude**, et *precision* par **précision**. Ce sont deux métriques différentes.

On regarde donc, pour la classe « **retard** » :

| Métrique | Question à laquelle elle répond |
|---|---|
| **Rappel** | Sur tous les vrais retards, combien le modèle en détecte-t-il ? (manquer un retard coûte cher) |
| **Précision** | Quand le modèle annonce un retard, a-t-il raison ? (une fausse alerte coûte aussi) |
| **F1** | Moyenne harmonique du rappel et de la précision : élevée seulement si les deux le sont |
| **ROC-AUC** | Probabilité qu'une commande en retard reçoive un score plus élevé qu'une commande à temps. Ne dépend pas du seuil, et vaut 0,5 pour un modèle au hasard |

**Toujours sur train seulement**, avec deux sources de prédictions honnêtes :
1. les prédictions **hors-pli** de la validation croisée regroupée par commande : chaque ligne est prédite par un modèle qui ne l'a pas vue ;
2. les prédictions de la **validation chronologique** (étape 4).

## 5.1 Prédictions hors-pli (validation croisée regroupée)

Pour chaque modèle, on récupère la **probabilité de retard** prédite pour chaque ligne de train, par le modèle du pli où elle était en validation.

In [ ]:
modeles_eval = {
    'Régression logistique': (modele_logistique, X_train_enc),
    'Forêt aléatoire': (foret, X_train_enc),
    'LightGBM': (lgbm, X_train_lgbm),
}

proba_cv = {}
for nom, (modele, X) in modeles_eval.items():
    proba_cv[nom] = cross_val_predict(clone(modele), X, y_train, groups=groupes, cv=cv,
                                      method='predict_proba', n_jobs=-1 if nom != 'LightGBM' else 1)[:, 1]
proba_cv['Naïf (toujours retard)'] = np.ones(len(y_train))

## 5.2 Tableau des métriques (seuil 0,5)

In [ ]:
eval_cv = pd.DataFrame({nom: scores(y_train, p) for nom, p in proba_cv.items()}).T
eval_cv = eval_cv[['exactitude', 'precision_retard', 'rappel_retard', 'f1', 'roc_auc']].round(3)
eval_cv.sort_values('roc_auc', ascending=False)

**Lecture :**
- Le modèle naïf a **57 % d'exactitude**, un **rappel de 100 %** et même le **meilleur F1**, car il annonce un retard partout. Son **ROC-AUC de 0,5** montre pourtant qu'il ne distingue rien. C'est l'exemple même du piège.
- Les vrais modèles ont une **précision élevée** (85 à 89 %) : leurs alertes sont fiables.
- Leur **rappel reste moyen** (57 à 61 %) : environ 4 retards sur 10 passent inaperçus.
- Le **ROC-AUC** est la mesure la plus fiable pour les classer : forêt aléatoire et LightGBM d'abord (environ 0,77), puis la régression logistique (0,74).

## 5.3 Matrices de confusion

Chaque case compte des lignes de train (prédictions hors-pli) :
- **en haut à droite** : fausses alertes (annoncées en retard, livrées à temps) ;
- **en bas à gauche** : retards manqués (annoncés à temps, livrés en retard).

In [ ]:
noms_modeles = ['Régression logistique', 'Forêt aléatoire', 'LightGBM']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, nom in zip(axes, noms_modeles):
    ConfusionMatrixDisplay.from_predictions(
        y_train, (proba_cv[nom] >= 0.5).astype(int),
        display_labels=['À temps', 'En retard'], ax=ax, colorbar=False, values_format=',d')
    ax.set_title(nom)
plt.tight_layout()

## 5.4 Courbes ROC et précision-rappel

- **Courbe ROC** : taux de retards détectés (rappel) en fonction du taux de fausses alertes, pour tous les seuils possibles. Plus la courbe monte vite vers le coin en haut à gauche, mieux c'est. La diagonale correspond au hasard.
- **Courbe précision-rappel** : ce qu'on gagne en rappel et ce qu'on perd en précision quand on baisse le seuil. La ligne horizontale à 0,57 correspond au modèle naïf.

Ces courbes montrent que le **seuil de 0,5 n'est qu'un choix parmi d'autres** : en le baissant, on détecte plus de retards, au prix de fausses alertes.

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for nom in noms_modeles:
    RocCurveDisplay.from_predictions(y_train, proba_cv[nom], name=nom, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_train, proba_cv[nom], name=nom, ax=axes[1])
axes[0].plot([0, 1], [0, 1], 'k--', label='Hasard (AUC = 0,5)')
axes[0].set_title('Courbe ROC (CV regroupée, train)')
axes[0].legend(loc='lower right')
axes[1].axhline(y_train.mean(), color='k', ls='--', label=f'Naïf ({y_train.mean():.2f})')
axes[1].set_title('Courbe précision-rappel (CV regroupée, train)')
axes[1].legend(loc='lower left')
axes[0].set_xlabel('Taux de fausses alertes'); axes[0].set_ylabel('Rappel (retards détectés)')
axes[1].set_xlabel('Rappel (retards détectés)'); axes[1].set_ylabel('Précision')
plt.tight_layout()

## 5.5 Les mêmes métriques en validation chronologique

On reprend les modèles entraînés jusqu'au 30 juin 2017 (étape 4) et on les évalue sur juillet 2017 à janvier 2018.

In [ ]:
y_valid = y_train[valid]
proba_chrono = {nom: modeles_chrono[nom][0].predict_proba(modeles_chrono[nom][1][valid])[:, 1]
                for nom in noms_modeles}
proba_chrono['Naïf (toujours retard)'] = np.ones(valid.sum())

eval_chrono = pd.DataFrame({nom: scores(y_valid, p) for nom, p in proba_chrono.items()}).T
eval_chrono = eval_chrono[['exactitude', 'precision_retard', 'rappel_retard', 'f1', 'roc_auc']].round(3)
eval_chrono.sort_values('roc_auc', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, nom in zip(axes, noms_modeles):
    ConfusionMatrixDisplay.from_predictions(
        y_valid, (proba_chrono[nom] >= 0.5).astype(int),
        display_labels=['À temps', 'En retard'], ax=ax, colorbar=False, values_format=',d')
    ax.set_title(f'{nom} (juil. 2017 - janv. 2018)')
plt.tight_layout()

## Conclusion de l'étape 5

- **L'exactitude trompe :** le modèle naïf atteint 57 % et le meilleur F1 sans rien apprendre. Il faut lire **ROC-AUC, rappel et précision ensemble**.
- **Meilleurs modèles :** forêt aléatoire et LightGBM, avec un ROC-AUC d'environ 0,77 en validation croisée et 0,78 en validation chronologique. Les deux validations concordent.
- **Point fort :** une **précision** élevée (85 à 89 %). Une alerte de retard est presque toujours justifiée.
- **Point faible :** le **rappel** (environ 60 %). Beaucoup de retards ne sont pas détectés au seuil 0,5.
- **Ce que montrent les courbes :** jusqu'à environ 60 % de rappel, la précision reste proche de 90 %. Au-delà, elle chute vite. Une partie des retards ressemble donc beaucoup aux commandes à temps avec les variables disponibles.
- **Piste suivante :** **choisir le seuil de décision** selon le coût d'un retard manqué face à celui d'une fausse alerte. Les courbes précision-rappel montrent le compromis possible.

Le jeu de test n'a toujours **pas** été utilisé.

---
# Étape 6 : ajuster les hyperparamètres et le seuil de décision

Deux réglages, toujours **sur train seulement** avec la validation croisée regroupée par commande :
1. **Hyperparamètres** : les « boutons » du modèle (nombre d'arbres, profondeur, taux d'apprentissage…).
2. **Seuil de décision** : à partir de quelle probabilité on annonce un retard. Jusqu'ici, on utilisait 0,5. On le choisit maintenant selon le **coût d'un retard manqué**.

Le test reste intact. Le modèle et le seuil choisis ici seront appliqués **une seule fois** au test, à l'étape finale.

## 6.1 Réglage des hyperparamètres

**Méthode : recherche aléatoire** (`RandomizedSearchCV`). On tire au hasard des combinaisons de paramètres, on évalue chacune par validation croisée regroupée et on garde la meilleure selon le **ROC-AUC**. C'est plus efficace qu'une grille complète quand il y a beaucoup de paramètres.

**Durée :** la recherche prend quelques minutes. Pour la sauter, mettre `FAIRE_RECHERCHE = False` : le notebook reprend alors les meilleurs paramètres déjà trouvés.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from scipy.stats import randint, uniform, loguniform

FAIRE_RECHERCHE = True

### Forêt aléatoire

Petite grille sur les paramètres qui comptent le plus :
- `min_samples_leaf` : taille minimale d'une feuille (plus grand = arbres plus simples) ;
- `max_features` : part des variables tirées à chaque division.

On utilise 100 arbres pendant la recherche pour aller plus vite.

In [ ]:
grille_foret = {'min_samples_leaf': [5, 20, 50], 'max_features': ['sqrt', 0.5]}

if FAIRE_RECHERCHE:
    recherche_foret = GridSearchCV(
        RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
        grille_foret, scoring='roc_auc', cv=cv, n_jobs=1)
    recherche_foret.fit(X_train_enc, y_train, groups=groupes)
    params_foret = recherche_foret.best_params_
    print('Meilleur ROC-AUC (CV) :', round(recherche_foret.best_score_, 4))
    display(pd.DataFrame(recherche_foret.cv_results_)[['params', 'mean_test_score']]
            .sort_values('mean_test_score', ascending=False).round(4))
else:
    params_foret = {'max_features': 0.5, 'min_samples_leaf': 50}
print('Paramètres retenus :', params_foret)

### LightGBM

Recherche aléatoire de 8 combinaisons sur 7 paramètres :
- `n_estimators`, `learning_rate` : nombre d'arbres et pas d'apprentissage ;
- `num_leaves`, `min_child_samples` : complexité de chaque arbre ;
- `subsample`, `colsample_bytree` : part des lignes et des colonnes vues par chaque arbre ;
- `reg_lambda` : régularisation, qui freine le sur-apprentissage.

In [ ]:
espace_lgbm = {
    'n_estimators': randint(200, 500),
    'learning_rate': loguniform(0.02, 0.2),
    'num_leaves': randint(15, 255),
    'min_child_samples': randint(10, 200),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.5, 0.5),
    'reg_lambda': loguniform(1e-3, 10),
}

if FAIRE_RECHERCHE:
    recherche_lgbm = RandomizedSearchCV(
        LGBMClassifier(subsample_freq=1, random_state=42, verbose=-1),
        espace_lgbm, n_iter=8, scoring='roc_auc', cv=cv, random_state=42, n_jobs=1)
    recherche_lgbm.fit(X_train_lgbm, y_train, groups=groupes)
    params_lgbm = recherche_lgbm.best_params_
    print('Meilleur ROC-AUC (CV) :', round(recherche_lgbm.best_score_, 4))
else:
    params_lgbm = {'colsample_bytree': 0.55, 'learning_rate': 0.058, 'min_child_samples': 126,
                   'n_estimators': 299, 'num_leaves': 118, 'reg_lambda': 0.68, 'subsample': 0.61}
print('Paramètres retenus :', {k: round(v, 3) if isinstance(v, float) else v for k, v in params_lgbm.items()})

### Bilan du réglage

In [ ]:
reglage = pd.DataFrame({
    'Avant réglage (étape 3)': [comparaison.loc['Forêt aléatoire', 'roc_auc'], comparaison.loc['LightGBM', 'roc_auc']],
    'Après réglage': [recherche_foret.best_score_ if FAIRE_RECHERCHE else np.nan,
                      recherche_lgbm.best_score_ if FAIRE_RECHERCHE else np.nan],
}, index=['Forêt aléatoire', 'LightGBM']).round(3)
reglage

**Lecture :** le réglage ne change presque rien, le ROC-AUC reste autour de **0,77**. Une recherche plus longue (15 combinaisons et jusqu'à 800 arbres, testée à part) donne aussi 0,770.

La limite ne vient donc pas des réglages du modèle, mais de **l'information contenue dans les 15 variables**. Pour progresser vraiment, il faudrait de nouvelles variables connues au moment de la commande (par exemple la charge de l'entrepôt, la distance ou la météo).

**Modèle retenu : LightGBM réglé.** Il est aussi bon que la forêt, plus rapide, et avait le meilleur rappel.

## 6.2 Choix du seuil selon le coût d'un retard manqué

Le modèle donne une **probabilité de retard**. Le seuil décide à partir de quand on annonce un retard :
- un **seuil bas** détecte plus de retards (rappel plus élevé), mais crée plus de fausses alertes ;
- un **seuil haut** donne moins de fausses alertes, mais laisse passer plus de retards.

**Le bon seuil dépend des coûts.** On note :
- **coût d'un retard manqué** (faux négatif) = `ratio_cout` × coût d'une fausse alerte ;
- **coût d'une fausse alerte** (faux positif) = 1.

Coût total = `ratio_cout` × (retards manqués) + (fausses alertes). On cherche le seuil qui **minimise ce coût**.

> **Hypothèse de travail : un retard manqué coûte 3 fois plus qu'une fausse alerte** (`ratio_cout = 3`). Ce chiffre est une hypothèse : **remplace-le par ton propre ratio** si tu connais les coûts réels (pénalités, client perdu, coût d'une action préventive…).

In [ ]:
ratio_cout = 3   # <- à adapter : coût d'un retard manqué / coût d'une fausse alerte

On calcule d'abord les probabilités **hors-pli** du LightGBM réglé, avec la même validation regroupée. Le seuil est donc choisi sur des prédictions honnêtes, sans toucher au test.

In [ ]:
lgbm_final = LGBMClassifier(subsample_freq=1, random_state=42, verbose=-1, **params_lgbm)
proba_final = cross_val_predict(lgbm_final, X_train_lgbm, y_train, groups=groupes, cv=cv,
                                method='predict_proba', n_jobs=1)[:, 1]
print('ROC-AUC hors-pli :', round(roc_auc_score(y_train, proba_final), 4))

In [ ]:
from sklearn.metrics import confusion_matrix

seuils = np.round(np.arange(0.05, 0.96, 0.01), 2)

def analyse_seuils(y, proba, ratio):
    lignes = []
    for s in seuils:
        tn, fp, fn, tp = confusion_matrix(y, (proba >= s).astype(int)).ravel()
        lignes.append({'seuil': s, 'retards_manques': fn, 'fausses_alertes': fp,
                       'rappel_retard': tp / (tp + fn), 'precision_retard': tp / (tp + fp) if tp + fp else 0,
                       'cout': ratio * fn + fp})
    return pd.DataFrame(lignes)

def meilleur_seuil(y, proba, ratio):
    t = analyse_seuils(y, proba, ratio)
    return t.loc[t['cout'].idxmin()]

### Seuil optimal pour plusieurs ratios de coût

Pour comparer, on calcule le seuil optimal selon que le retard manqué coûte 1 ; 1,5 ; 2 ; 3 ou 5 fois une fausse alerte. Le coût est comparé à celui du seuil par défaut (0,5).

In [ ]:
lignes = []
for r in [1, 1.5, 2, 3, 5]:
    t = analyse_seuils(y_train, proba_final, r)
    m = t.loc[t['cout'].idxmin()]
    cout_05 = t.loc[t['seuil'] == 0.5, 'cout'].iloc[0]
    lignes.append({'ratio_cout': r, 'seuil_optimal': m['seuil'],
                   'rappel_retard': m['rappel_retard'], 'precision_retard': m['precision_retard'],
                   'retards_manques': int(m['retards_manques']), 'fausses_alertes': int(m['fausses_alertes']),
                   'economie_vs_seuil_0.5': f"{1 - m['cout'] / cout_05:.1%}"})
seuils_par_ratio = pd.DataFrame(lignes).round(3)
seuils_par_ratio

**Lecture, à ne pas manquer :**
- Avec un **ratio de 1** (les deux erreurs coûtent pareil), le seuil optimal est d'environ **0,6** : on n'alerte que sur les commandes très probablement en retard, avec une précision d'environ 89 %.
- Dès que le ratio dépasse **environ 1,6**, le seuil optimal tombe sous **0,25** : le modèle annonce **presque toutes les commandes en retard** (rappel d'environ 100 %, précision d'environ 59 %, à peine mieux que le modèle naïf).

**Pourquoi ?** Environ 40 % des retards ressemblent trait pour trait aux commandes à temps avec nos 15 variables : c'est la cassure de la courbe précision-rappel vers 60 % de rappel (étape 5). Pour attraper ces retards-là, il faut alerter presque tout le monde. Dès qu'un retard manqué coûte plus d'environ 1,6 fausse alerte, **alerter tout le monde revient moins cher** que de laisser passer ces retards.

La cellule suivante cherche le **ratio de bascule** : au-delà, le modèle ne trie plus vraiment les commandes.

In [ ]:
bascule = []
for r in np.round(np.arange(1, 3.01, 0.1), 1):
    m = meilleur_seuil(y_train, proba_final, r)
    bascule.append({'ratio_cout': r, 'seuil_optimal': m['seuil'], 'rappel_retard': round(m['rappel_retard'], 3),
                    'precision_retard': round(m['precision_retard'], 3)})
bascule = pd.DataFrame(bascule)
ratio_bascule = bascule.loc[bascule['seuil_optimal'] <= 0.25, 'ratio_cout'].min()
print('Ratio de bascule (le seuil optimal passe sous 0,25) :', ratio_bascule)
bascule

### Avec l'hypothèse retenue

In [ ]:
table = analyse_seuils(y_train, proba_final, ratio_cout)
choix = table.loc[table['cout'].idxmin()]
seuil_retenu = float(choix['seuil'])

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(table['seuil'], table['cout'])
ax[0].axvline(seuil_retenu, color='r', ls='--', label=f'Seuil optimal = {seuil_retenu:.2f}')
ax[0].axvline(0.5, color='grey', ls=':', label='Seuil par défaut = 0,5')
ax[0].set_xlabel('Seuil'); ax[0].set_ylabel('Coût total'); ax[0].legend()
ax[0].set_title(f'Coût total selon le seuil (ratio = {ratio_cout})')
ax[1].plot(table['seuil'], table['rappel_retard'], label='Rappel (retards détectés)')
ax[1].plot(table['seuil'], table['precision_retard'], label='Précision des alertes')
ax[1].axvline(seuil_retenu, color='r', ls='--')
ax[1].set_xlabel('Seuil'); ax[1].set_ylim(0, 1.05); ax[1].legend()
ax[1].set_title('Rappel et précision selon le seuil')
plt.tight_layout()

print(f'Seuil retenu : {seuil_retenu:.2f}')
print(f"Rappel : {choix['rappel_retard']:.3f} | Précision : {choix['precision_retard']:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, s, titre in [(axes[0], 0.5, 'Seuil 0,5'), (axes[1], seuil_retenu, f'Seuil {seuil_retenu:.2f} (ratio {ratio_cout})')]:
    ConfusionMatrixDisplay.from_predictions(y_train, (proba_final >= s).astype(int),
        display_labels=['À temps', 'En retard'], ax=ax, colorbar=False, values_format=',d')
    ax.set_title(titre)
plt.tight_layout()

### Vérification en validation chronologique

Le seuil a été choisi sur la validation croisée. On vérifie qu'il donne le même effet sur les mois récents (juillet 2017 à janvier 2018), avec un modèle entraîné sur le passé.

In [ ]:
lgbm_chrono = LGBMClassifier(subsample_freq=1, random_state=42, verbose=-1, **params_lgbm)
lgbm_chrono.fit(X_train_lgbm[apprend], y_train[apprend])
proba_recent = lgbm_chrono.predict_proba(X_train_lgbm[valid])[:, 1]

verif = pd.DataFrame([
    {'seuil': s, **{k: v for k, v in scores(y_valid, proba_recent).items() if k == 'roc_auc'},
     'rappel_retard': recall_score(y_valid, (proba_recent >= s).astype(int)),
     'precision_retard': precision_score(y_valid, (proba_recent >= s).astype(int)),
     'f1': f1_score(y_valid, (proba_recent >= s).astype(int))}
    for s in [0.5, seuil_retenu]]).round(3)
verif

## Conclusion de l'étape 6

- **Hyperparamètres :** le réglage ne gagne presque rien (ROC-AUC d'environ 0,77). La limite vient des variables, pas du modèle.
- **Seuil :** c'est le réglage qui change vraiment le comportement du modèle, et **tout dépend du ratio de coût** :
  - **ratio inférieur au ratio de bascule** (environ 1,6) : le modèle trie utilement les commandes, avec un seuil entre 0,5 et 0,6 ;
  - **ratio supérieur** (dont l'hypothèse de 3) : le seuil optimal revient à **alerter presque toutes les commandes**. Le modèle n'apporte alors presque rien par rapport à « tout traiter comme à risque ».
- **En pratique**, avec un retard manqué coûteux, une stratégie à **deux niveaux** est plus utile qu'un seuil unique : une action forte sur les commandes à haute probabilité (au-dessus de 0,6, alertes justes à environ 89 %) et une action légère et peu coûteuse sur les autres.
- **Modèle final retenu :** LightGBM réglé. Le seuil dépend de `ratio_cout`, à fixer selon tes coûts réels.

Le jeu de test n'a toujours **pas** été utilisé. **Prochaine et dernière étape :** entraîner ce modèle sur tout train et l'évaluer **une seule fois** sur le test, avec le seuil retenu.

---
# Étape 7 : interpréter le modèle

**But :** vérifier que le modèle retenu (LightGBM réglé) s'appuie sur des raisons **logiques**, et non sur des artefacts ou des fuites.

**Données utilisées : train seulement.** On reprend le modèle entraîné sur les commandes jusqu'au 30 juin 2017 (étape 6) et on l'explique sur les mois récents de train (juillet 2017 à janvier 2018), qu'il n'a pas vus. Le test reste intact.

Deux outils complémentaires :
1. **Importance par permutation** : on mélange une variable au hasard et on mesure la perte de ROC-AUC. Si le score chute, la variable compte. On travaille sur les **15 variables d'origine**, en mélangeant toutes les colonnes one-hot d'une variable ensemble.
2. **SHAP** : pour chaque commande, la contribution de chaque variable à la prédiction (en log-odds : positif = pousse vers le retard, négatif = vers l'à-temps). Installation si besoin : `pip install shap`.

## 7.1 Importance par permutation (15 variables d'origine)

In [ ]:
from sklearn.metrics import roc_auc_score

X_recent = X_train[valid]          # 15 variables brutes, mois récents de train
y_recent = y_train[valid]

def predire(X_brut):
    return lgbm_chrono.predict_proba(encoder(X_brut).rename(columns=noms_propres))[:, 1]

score_ref = roc_auc_score(y_recent, predire(X_recent))
print('ROC-AUC de référence (mois récents de train) :', round(score_ref, 4))

rng = np.random.default_rng(42)
pertes = {}
for var in variables:
    essais = []
    for _ in range(3):                       # 3 mélanges par variable, puis moyenne
        X_melange = X_recent.copy()
        X_melange[var] = rng.permutation(X_melange[var].values)
        essais.append(score_ref - roc_auc_score(y_recent, predire(X_melange)))
    pertes[var] = np.mean(essais)

importance_perm = pd.Series(pertes).sort_values(ascending=False)
importance_perm.round(4)

In [ ]:
importance_perm.sort_values().plot.barh(figsize=(8, 5))
plt.xlabel('Perte de ROC-AUC quand la variable est mélangée')
plt.title('Importance par permutation (mois récents de train)')
plt.tight_layout()

**Lecture :** trois variables portent presque toute l'information :
- **`Shipping Mode`** (mode d'expédition) ;
- **`Days for shipment (scheduled)`** (délai promis, lié 1 pour 1 au mode d'expédition) ;
- **`order_hour`** (heure de la commande).

Les 12 autres variables (produit, client, région, prix, remise, mois…) ont une importance **quasi nulle** : les mélanger ne change presque rien. Le modèle ne s'appuie donc sur aucune variable suspecte.

## 7.2 SHAP : vue d'ensemble

On calcule les valeurs SHAP sur un échantillon de 2 000 lignes des mois récents (pour la rapidité).

Dans le graphique ci-dessous, chaque point est une commande :
- **position horizontale** : effet de la variable sur la prédiction (à droite = pousse vers le retard) ;
- **couleur** : valeur de la variable (rouge = élevée, bleu = basse ; pour une colonne one-hot, rouge = 1).

In [ ]:
import shap

echantillon = X_recent.sample(2000, random_state=42)
X_ech = encoder(echantillon)                      # noms lisibles

explicateur = shap.TreeExplainer(lgbm_chrono)
valeurs_shap = explicateur(X_ech.rename(columns=noms_propres))
valeurs_shap.feature_names = list(X_ech.columns)  # on remet les noms lisibles

shap.plots.beeswarm(valeurs_shap, max_display=12)

## 7.3 SHAP : effet des variables clés

**Heure de commande.** La couleur indique si la commande est en *Same Day* (rouge) ou non (bleu).

In [ ]:
shap.plots.scatter(valeurs_shap[:, 'order_hour'], color=valeurs_shap[:, 'Shipping Mode_Same Day'])

**Délai promis.** La couleur indique l'heure de commande.

In [ ]:
shap.plots.scatter(valeurs_shap[:, 'Days for shipment (scheduled)'], color=valeurs_shap[:, 'order_hour'])

## 7.4 Vérification dans les données brutes (train)

Pour confirmer que ces effets sont réels et logiques, on regarde directement les taux de retard dans train, par mode d'expédition et par moment de la commande (avant ou après midi).

In [ ]:
verif_logique = train.assign(moment=np.where(train['order_hour'] < 12, 'avant midi', 'après midi')) \
    .pivot_table(index='Shipping Mode', columns='moment', values=cible, aggfunc='mean').round(3)
verif_logique['délai promis (jours)'] = train.groupby('Shipping Mode')['Days for shipment (scheduled)'].first()
verif_logique[['délai promis (jours)', 'avant midi', 'après midi']]

**Le modèle a retrouvé des règles logiques :**

| Mode | Délai promis | Taux de retard | Explication |
|---|---|---|---|
| **First Class** | 1 jour | **100 %** | L'expédition réelle prend toujours 2 jours : la promesse n'est jamais tenue. |
| **Same Day** | 0 jour | **0 % avant midi, 100 % après midi** | Une commande passée l'après-midi part le lendemain, donc en retard. |
| **Second Class** | 2 jours | environ 80 % | La durée réelle varie de 2 à 6 jours, sans lien avec les autres variables. |
| **Standard Class** | 4 jours | environ 40 % | Idem : durée réelle de 2 à 6 jours. |

C'est **cohérent avec le métier** : un délai promis court est plus difficile à tenir, et l'heure limite de préparation compte pour la livraison le jour même.

**Cela explique aussi le plafond du ROC-AUC (environ 0,77).** Pour *Second Class* et *Standard Class* (plus de 75 % des commandes), le retard semble dépendre du hasard : aucune des variables disponibles ne permet de le prévoir. C'est la cassure à 60 % de rappel observée à l'étape 5.

## 7.5 Exemples individuels

On explique trois commandes des mois récents. Le graphique part de la **prédiction moyenne** (E[f(x)]) et montre comment chaque variable pousse vers le retard (rouge) ou vers l'à-temps (bleu), jusqu'à la prédiction finale (f(x), en log-odds).

In [ ]:
def exemple(masque, titre):
    i = np.where(masque)[0][0]
    proba = 1 / (1 + np.exp(-valeurs_shap[i].values.sum() - valeurs_shap[i].base_values))
    print(f'{titre} | probabilité de retard prédite : {proba:.2f} | réel : {"retard" if y_recent.loc[echantillon.index[i]] == 1 else "à temps"}')
    shap.plots.waterfall(valeurs_shap[i], max_display=8)

mode = echantillon['Shipping Mode'].values
heure = echantillon['order_hour'].values
exemple((mode == 'Same Day') & (heure >= 12), 'Exemple 1 : Same Day, commandée l\'après-midi')

In [ ]:
exemple((mode == 'Same Day') & (heure < 12), 'Exemple 2 : Same Day, commandée le matin')

In [ ]:
exemple(mode == 'Standard Class', 'Exemple 3 : Standard Class')

## Conclusion de l'étape 7

- **Le modèle reste logique.** Il s'appuie sur le **mode d'expédition**, le **délai promis** et l'**heure de commande**, avec des effets qui ont un sens métier (promesse First Class intenable, heure limite pour le Same Day).
- **Aucune variable suspecte.** Les identifiants et les colonnes qui fuient la réponse ont été retirés (étape 1), et les autres variables ont une importance quasi nulle.
- **Limite claire :** pour *Second Class* et *Standard Class*, le retard n'est pas prévisible avec ces données. Pour progresser, il faudrait de nouvelles informations sur l'expédition (transporteur, entrepôt, distance, charge).

Le jeu de test n'a toujours **pas** été utilisé.

---
# Étape 8 : modèle final, évaluation sur le test et export pour l'application

C'est la **seule** étape qui utilise le jeu de test. Elle se déroule ainsi :
1. **Choisir le ratio de coût** et donc le seuil de décision (sur train).
2. **Réentraîner** le modèle retenu (LightGBM réglé) sur **tout** train.
3. **Évaluer une seule fois** sur le test. On ne revient plus en arrière après : le score obtenu est l'estimation honnête de la performance future.
4. **Enregistrer** le modèle, l'encodage et le seuil dans un fichier unique, à remettre à la personne qui développe l'application.

## 8.1 Le choix du ratio de coût

### Ce que c'est

Le modèle donne une **probabilité de retard**. Pour décider « retard » ou « à temps », il faut un **seuil**, et ce seuil dépend de ce que coûte chaque type d'erreur :

$$\text{ratio\_cout} = \frac{\text{coût d'un retard manqué (faux négatif)}}{\text{coût d'une fausse alerte (faux positif)}}$$

*Exemple : si rater un retard coûte 30 $ (client mécontent, pénalité) et qu'une fausse alerte coûte 10 $ (appel ou suivi inutile), alors ratio_cout = 30 / 10 = 3.*

### Pourquoi ce n'est pas un choix « technique »

**Les données ne peuvent pas fixer ce ratio.** Elles disent seulement *qui* risque d'être en retard. Combien coûte une erreur dépend de l'**entreprise** : pénalités contractuelles, valeur du client, coût de l'action préventive. C'est une **décision métier**, à prendre avec les responsables logistiques.

### Le choix retenu : ratio = 1

En l'absence de coûts réels connus, on retient **ratio_cout = 1**, le choix **neutre** :
- les deux erreurs pèsent autant, sans préférence pour l'une ou l'autre ;
- le seuil optimal est alors d'environ **0,6** ;
- c'est la zone où le modèle **trie vraiment** les commandes : quand il annonce un retard, il a raison environ 9 fois sur 10.

### La bascule à 1,6 (étape 6)

| Ratio | Seuil optimal | Comportement |
|---|---|---|
| de 1 à 1,5 | environ 0,6 | le modèle trie les commandes : alertes fiables (précision d'environ 89 %), environ 59 % des retards détectés |
| 1,6 et plus | sous 0,25 | le modèle alerte **presque toutes** les commandes (précision d'environ 59 %, à peine mieux que « tout est en retard ») |

Cette bascule vient des données : pour *Second Class* et *Standard Class*, le retard est imprévisible (étape 7). Dès qu'un retard manqué coûte plus d'environ 1,6 fausse alerte, il devient moins cher d'alerter tout le monde que de laisser passer ces retards-là. **Un ratio de 1,6 ou plus rend donc le modèle presque inutile**, et une règle simple (« toutes les commandes sont à risque ») ferait aussi bien.

### Comment changer le ratio

Modifier la valeur de `ratio_final` dans la cellule ci-dessous, puis réexécuter l'étape 8. Le seuil est recalculé automatiquement sur train, puis enregistré avec le modèle. Le développeur n'a rien à changer dans son code.

In [ ]:
ratio_final = 1   # <- décision métier : coût d'un retard manqué / coût d'une fausse alerte

choix_final = meilleur_seuil(y_train, proba_final, ratio_final)
seuil_final = float(choix_final['seuil'])
print(f'Ratio de coût retenu : {ratio_final}')
print(f'Seuil de décision    : {seuil_final:.2f}')
print(f"Attendu (CV sur train) -> rappel : {choix_final['rappel_retard']:.3f} | précision : {choix_final['precision_retard']:.3f}")

## 8.2 Réentraînement sur tout train

Pour l'application, on regroupe **tout** dans un seul objet scikit-learn (`Pipeline`) :
1. l'**encodage one-hot** des 7 variables catégorielles ;
2. le **modèle LightGBM** réglé (étape 6).

Ainsi, l'application fournit les **15 variables brutes**, telles qu'elles sont dans les données, et le pipeline se charge du reste. On évite les erreurs d'encodage côté application.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

modele_final = Pipeline([
    ('encodage', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), variables_cat),
        ('num', 'passthrough', variables_num),
    ])),
    ('lgbm', LGBMClassifier(subsample_freq=1, random_state=42, verbose=-1, **params_lgbm)),
])
modele_final.fit(X_train, y_train)    # 15 variables brutes, tout train
print('Modèle entraîné sur', len(X_train), 'lignes')

## 8.3 Évaluation finale sur le test (une seule fois)

On applique le modèle et le seuil, **choisis uniquement sur train**, aux 25 915 lignes du test. Aucun réglage ne sera fait après avoir vu ces résultats.

In [ ]:
test = pd.read_csv('test.csv', sep=';')
X_test_brut, y_test = test[variables], test[cible]

proba_test = modele_final.predict_proba(X_test_brut)[:, 1]

resultat_test = pd.DataFrame([
    {'evaluation': f'Test, seuil {seuil_final:.2f} (ratio {ratio_final})', **scores_seuil}
    for scores_seuil in [dict(zip(['exactitude', 'precision_retard', 'rappel_retard', 'f1'],
        [accuracy_score(y_test, proba_test >= seuil_final), precision_score(y_test, proba_test >= seuil_final),
         recall_score(y_test, proba_test >= seuil_final), f1_score(y_test, proba_test >= seuil_final)]))]
])
resultat_test['roc_auc'] = roc_auc_score(y_test, proba_test)

attendu = pd.DataFrame([{'evaluation': f'Attendu (CV sur train), seuil {seuil_final:.2f}',
    'exactitude': accuracy_score(y_train, proba_final >= seuil_final),
    'precision_retard': choix_final['precision_retard'], 'rappel_retard': choix_final['rappel_retard'],
    'f1': f1_score(y_train, proba_final >= seuil_final), 'roc_auc': roc_auc_score(y_train, proba_final)}])

pd.concat([attendu, resultat_test]).set_index('evaluation').round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, (proba_test >= seuil_final).astype(int),
    display_labels=['À temps', 'En retard'], ax=axes[0], colorbar=False, values_format=',d')
axes[0].set_title(f'Test : matrice de confusion (seuil {seuil_final:.2f})')
RocCurveDisplay.from_predictions(y_test, proba_test, name='LightGBM final', ax=axes[1])
axes[1].plot([0, 1], [0, 1], 'k--', label='Hasard')
axes[1].set_xlabel('Taux de fausses alertes'); axes[1].set_ylabel('Rappel (retards détectés)')
axes[1].set_title('Test : courbe ROC'); axes[1].legend(loc='lower right')
plt.tight_layout()

**Lecture :** au seuil choisi, le test donne **les mêmes résultats que prévu** : précision d'environ 0,89 à 0,90, rappel d'environ 0,58 et F1 de 0,71. Le seuil se généralise bien.

**Attention au ROC-AUC du test, plus élevé que prévu.** Le découpage train/test du début était aléatoire par ligne : beaucoup de commandes ont des articles dans train **et** dans test. Pour ces lignes, le modèle a déjà vu une commande identique (même mode, même heure), ce qui gonfle le ROC-AUC. On le vérifie en séparant les lignes du test selon que leur commande apparaît ou non dans train :

In [ ]:
commandes_train = set(train['Order Id'])
deja_vue = test['Order Id'].isin(commandes_train).values

detail = pd.DataFrame([{
    'lignes du test': nom, 'nombre': int(m.sum()),
    'roc_auc': roc_auc_score(y_test[m], proba_test[m]),
    'precision_retard': precision_score(y_test[m], proba_test[m] >= seuil_final),
    'rappel_retard': recall_score(y_test[m], proba_test[m] >= seuil_final)}
    for nom, m in [('commande aussi présente dans train', deja_vue),
                   ('commande absente de train (vraiment nouvelle)', ~deja_vue)]]).set_index('lignes du test').round(3)
detail

Sur les commandes **vraiment nouvelles**, le ROC-AUC retombe à environ **0,76**, en accord avec la validation regroupée (0,77). La précision et le rappel au seuil restent stables.

**Chiffres à communiquer comme performance du modèle :**
- **ROC-AUC d'environ 0,77** (validation regroupée et chronologique, confirmée sur les commandes nouvelles du test) ;
- au seuil de 0,61 : **précision d'environ 0,89** (une alerte sur 10 est fausse) et **rappel d'environ 0,58** (58 % des retards détectés).

Le 0,805 du test global est **optimiste** et ne doit pas être annoncé comme performance attendue.

## 8.4 Enregistrement du modèle pour l'application

On enregistre un **seul fichier**, `modele_retard.joblib`, qui contient :
- le **pipeline** (encodage + LightGBM) ;
- le **seuil** de décision et le **ratio** qui l'a produit ;
- la **liste et l'ordre des 15 variables** attendues ;
- les **versions** des bibliothèques (l'application doit utiliser les mêmes).

On écrit aussi `modele_retard_info.json`, une fiche lisible qui décrit les entrées, les catégories connues et les performances.

In [ ]:
import joblib, json, sklearn, lightgbm, platform
from datetime import date

paquet = {
    'pipeline': modele_final,
    'seuil': seuil_final,
    'ratio_cout': ratio_final,
    'variables': variables,
    'variables_categorielles': variables_cat,
    'variables_numeriques': variables_num,
    'versions': {'python': platform.python_version(), 'scikit-learn': sklearn.__version__,
                 'lightgbm': lightgbm.__version__, 'pandas': pd.__version__},
    'date': str(date.today()),
}
joblib.dump(paquet, 'modele_retard.joblib')

fiche = {
    'description': 'Prédiction du risque de retard de livraison (Late_delivery_risk) au moment de la commande',
    'sortie': {'probabilite_retard': 'entre 0 et 1',
               'retard_prevu': f'1 si probabilite_retard >= {seuil_final:.2f}, sinon 0'},
    'seuil': seuil_final, 'ratio_cout': ratio_final,
    'entrees': {
        **{v: {'type': 'texte', 'valeurs_connues': sorted(X_train[v].unique().tolist())} for v in variables_cat},
        **{v: {'type': 'nombre', 'min_vu': float(X_train[v].min()), 'max_vu': float(X_train[v].max())} for v in variables_num},
    },
    'performance_test': resultat_test.drop(columns='evaluation').round(3).iloc[0].to_dict(),
    'versions': paquet['versions'],
}
with open('modele_retard_info.json', 'w', encoding='utf-8') as f:
    json.dump(fiche, f, ensure_ascii=False, indent=2)
print('Fichiers écrits : modele_retard.joblib, modele_retard_info.json')

### Vérification : recharger le fichier et prédire comme l'application

On recharge le fichier comme le ferait l'application, puis on prédit une nouvelle commande décrite par ses 15 variables.

In [ ]:
paquet_charge = joblib.load('modele_retard.joblib')

nouvelle_commande = pd.DataFrame([{
    'Type': 'DEBIT',
    'Shipping Mode': 'Same Day',
    'Customer Segment': 'Consumer',
    'Category Name': 'Cleats',
    'Department Name': 'Apparel',
    'Market': 'LATAM',
    'Order Region': 'Central America',
    'Days for shipment (scheduled)': 0,
    'Order Item Product Price': 59.99,
    'Order Item Quantity': 2,
    'Order Item Discount Rate': 0.05,
    'Order Item Profit Ratio': 0.25,
    'order_month': 11,
    'order_weekday': 2,
    'order_hour': 15,
}])

p = paquet_charge['pipeline'].predict_proba(nouvelle_commande[paquet_charge['variables']])[:, 1][0]
print(f"Probabilité de retard : {p:.2f} -> {'RETARD prévu' if p >= paquet_charge['seuil'] else 'à temps'}")

# Contrôle : le modèle rechargé donne exactement les mêmes probabilités sur le test
assert np.allclose(paquet_charge['pipeline'].predict_proba(X_test_brut)[:, 1], proba_test)

## 8.5 Guide pour le développeur de l'application

**Fichiers à transmettre :** `modele_retard.joblib`, `modele_retard_info.json` et `predire.py`.

**Installation (mêmes versions que l'entraînement, voir `versions` dans le fichier info) :**
```
pip install pandas scikit-learn lightgbm joblib
```

**Entrées : les 15 variables d'une ligne de commande**, avec les noms exacts ci-dessous :

| Variable | Type | Exemple | Remarque |
|---|---|---|---|
| `Type` | texte | `DEBIT` | DEBIT, TRANSFER, PAYMENT, CASH |
| `Shipping Mode` | texte | `Standard Class` | Standard Class, Second Class, First Class, Same Day |
| `Customer Segment` | texte | `Consumer` | Consumer, Corporate, Home Office |
| `Category Name` | texte | `Cleats` | liste dans le fichier info |
| `Department Name` | texte | `Apparel` | liste dans le fichier info |
| `Market` | texte | `LATAM` | LATAM, Europe, Pacific Asia, USCA, Africa |
| `Order Region` | texte | `Central America` | liste dans le fichier info |
| `Days for shipment (scheduled)` | entier | `4` | délai promis : 0 Same Day, 1 First, 2 Second, 4 Standard |
| `Order Item Product Price` | nombre | `59.99` | prix unitaire |
| `Order Item Quantity` | entier | `2` | |
| `Order Item Discount Rate` | nombre | `0.05` | entre 0 et 1 |
| `Order Item Profit Ratio` | nombre | `0.25` | |
| `order_month` | entier | `11` | 1 à 12 |
| `order_weekday` | entier | `2` | 0 = lundi … 6 = dimanche |
| `order_hour` | entier | `15` | 0 à 23, heure de la commande |

**Sortie :** une probabilité de retard (0 à 1) et une décision (1 = retard prévu si probabilité ≥ seuil).

**Points d'attention :**
- Une catégorie inconnue (nouveau marché, nouvelle catégorie) **ne provoque pas d'erreur** : elle est simplement ignorée par l'encodage. Il vaut quand même la peine de la signaler.
- Toutes les variables sont **connues au moment de la commande** : aucune information après expédition n'est nécessaire.
- Pour changer le seuil (nouveau ratio de coût), il suffit de réexécuter l'étape 8 : le nouveau seuil est enregistré dans le fichier, sans changer le code de l'application.

## Conclusion de l'étape 8

- Le modèle final est un **LightGBM réglé**, entraîné sur tout train, avec un seuil choisi pour un **ratio de coût de 1** (choix neutre, à remplacer par les coûts réels).
- Sa performance sur le **test**, mesurée une seule fois, est dans le tableau de la section 8.3.
- **Livrables pour l'application :** `modele_retard.joblib`, `modele_retard_info.json`, `predire.py`.